In [1]:
print("hello")

hello


In [2]:
pip install transformers peft bitsandbytes trl accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 41.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# LLM for fine tuning

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-Coder-3B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

# DATASET ALTERATION FOR SFT NOW

In [15]:
import json
from pathlib import Path
from collections import defaultdict
import random


INPUT_PATH = "/kaggle/input/datasets/ibtihussain/pref-pair-sft-dpo/preference_pairs.jsonl"
TRAIN_OUTPUT_PATH = "/kaggle/working/sft_train.jsonl"
HOLDOUT_OUTPUT_PATH = "/kaggle/working/sft_holdout.jsonl"  

SEED = 10
HOLDOUT_FRACTION = 0.2 # 20% MEANS 6 OUT OF 30 TASKS IDs WILL BE FOR EVALS


all_pairs = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        all_pairs.append(json.loads(line))

print(f"Loaded {len(all_pairs)} total preference pairs")

Loaded 112 total preference pairs


In [16]:
# --- Split by task_id (not by row) to avoid leakage across train/eval ---
unique_task_ids = sorted(set(p["task_id"] for p in all_pairs))
print(f"Unique task_ids: {len(unique_task_ids)}")

random.seed(SEED)
shuffled_task_ids = unique_task_ids.copy()
random.shuffle(shuffled_task_ids)

num_holdout = max(1, round(len(shuffled_task_ids) * HOLDOUT_FRACTION))
holdout_task_ids = set(shuffled_task_ids[:num_holdout])
train_task_ids = set(shuffled_task_ids[num_holdout:])

print(f"Holdout task_ids ({len(holdout_task_ids)}): {sorted(holdout_task_ids)}")
print(f"Train task_ids ({len(train_task_ids)}): {sorted(train_task_ids)}")

Unique task_ids: 30
Holdout task_ids (6): ['task_05', 'task_08', 'task_10', 'task_18', 'task_21', 'task_23']
Train task_ids (24): ['task_01', 'task_02', 'task_03', 'task_04', 'task_06', 'task_07', 'task_09', 'task_11', 'task_12', 'task_13', 'task_14', 'task_15', 'task_16', 'task_17', 'task_19', 'task_20', 'task_22', 'task_24', 'task_25', 'task_26', 'task_27', 'task_28', 'task_29', 'task_30']


In [18]:
# Build SFT examples: prompt + chosen only (drop rejected for this stage)
def build_sft_record(pair):
    return {
        "task_id": pair["task_id"],
        "tool": pair["tool"],
        "tier": pair["tier"],
        "messages": [
            {"role": "user", "content": pair["prompt"]},
            {"role": "assistant", "content": pair["chosen"]},
        ],
    }

train_records = [build_sft_record(p) for p in all_pairs if p["task_id"] in train_task_ids]
holdout_records = [build_sft_record(p) for p in all_pairs if p["task_id"] in holdout_task_ids]

print(f"\nSFT train examples: {len(train_records)}")
print(f"SFT holdout examples: {len(holdout_records)}")



SFT train examples: 92
SFT holdout examples: 20


In [19]:
with open(TRAIN_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in train_records:
        f.write(json.dumps(rec) + "\n")

with open(HOLDOUT_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in holdout_records:
        f.write(json.dumps(rec) + "\n")

print(f"\nSaved train set to {TRAIN_OUTPUT_PATH}")
print(f"Saved holdout set to {HOLDOUT_OUTPUT_PATH}")


Saved train set to /kaggle/working/sft_train.jsonl
Saved holdout set to /kaggle/working/sft_holdout.jsonl
